In [1]:
# Load Data
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\Asus\Downloads\dirty_cafe_sales.csv')
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [2]:
# Total Rows & Columns
df.shape

(10000, 8)

In [3]:
# Structure Columns
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [4]:
# Total Missing Value
cols = [
    'Item',
    'Quantity',
    'Price Per Unit',
    'Total Spent',
    'Payment Method',
    'Location',
    'Transaction Date'
]

summary = pd.DataFrame({
    'UNKNOWN':(df[cols]=='UNKNOWN').sum(),
    'ERROR':(df[cols]=='ERROR').sum(),
    'Blank':df[cols].isna().sum()+(df[cols]=='').sum()
}).T

summary

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
UNKNOWN,344,171,164,165,293,338,159
ERROR,292,170,190,164,306,358,142
Blank,333,138,179,173,2579,3265,159


In [5]:
# Chek Duplicate
df.duplicated().sum()

np.int64(0)

In [6]:
# Categorical Value
categorical_cols = [
    'Item',
    'Payment Method',
    'Location'
]

for col in categorical_cols:
    print(f'\n{col}')
    print(df[col].unique())


Item
<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str

Payment Method
<StringArray>
['Credit Card', 'Cash', 'UNKNOWN', 'Digital Wallet', 'ERROR', nan]
Length: 6, dtype: str

Location
<StringArray>
['Takeaway', 'In-store', 'UNKNOWN', nan, 'ERROR']
Length: 5, dtype: str


In [7]:
# Standardize Categorical Columns
for col in categorical_cols:
    df[col] = (
        df[col]
        .replace(['UNKNOWN', 'ERROR', ''], 'Unknown')
        .fillna('Unknown')
    )

In [8]:
# Standardize Numerical Colums
numeric_cols = [
    'Quantity',
    'Price Per Unit',
    'Total Spent'
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .replace(['UNKNOWN', 'ERROR', ''], np.nan)
    )

In [9]:
# Standardize Date Columns
df['Transaction Date'] = (
    df['Transaction Date']
    .replace(['UNKNOWN', 'ERROR', ''], np.nan)
)

In [10]:
# Change Data Type
df['Quantity'] = df['Quantity'].astype('Int64')

df[['Price Per Unit', 'Total Spent']] = (
    df[['Price Per Unit', 'Total Spent']]
    .astype('float64')
)

df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])

In [11]:
#  Fix Item Using Price Mapping 
item_map = {
    2: 'Coffee',
    1: 'Cookie',
    5: 'Salad',
    1.5: 'Tea'
}

item_mask = df['Item'] == 'Unknown'

df.loc[item_mask, 'Item'] = (
    df.loc[item_mask, 'Price Per Unit']
    .map(item_map)
    .fillna('Unknown')
)

In [12]:
# Fix Price Using Item Mapping 
price_map = {
    'Coffee':2,
    'Cake':3,
    'Cookie':1,
    'Salad':5,
    'Smoothie':4,
    'Sandwich':4,
    'Juice':3,
    'Tea':1.5,
}

price_mask = df['Price Per Unit'].isna()

df.loc[price_mask, 'Price Per Unit'] = (
    df.loc[price_mask, 'Item']
    .map(price_map)
)

In [13]:
# Fix Quantity Using Spent and Price
quantity_mask = (
    df['Quantity'].isna()&
    df['Price Per Unit'].notna()&
    df['Total Spent'].notna()
)

df.loc[quantity_mask, 'Quantity'] = (
    df.loc[quantity_mask, 'Total Spent'] / 
    df.loc[quantity_mask, 'Price Per Unit']
)

In [14]:
# Fix Spent Using Quantity and Price
spent_mask = (
    df['Total Spent'].isna()&
    df['Quantity'].notna()&
    df['Price Per Unit'].notna()
)

df.loc[spent_mask, 'Total Spent'] = (
    df.loc[spent_mask, 'Quantity'] *
    df.loc[spent_mask, 'Price Per Unit']
)

In [15]:
# Fix Price Using Spent and Quantity
price_mask_2 = (
    df['Price Per Unit'].isna()&
    df['Total Spent'].notna()&
    df['Quantity'].notna()
)

df.loc[price_mask_2, 'Price Per Unit'] = (
    df.loc[price_mask_2, 'Total Spent'] /
    df.loc[price_mask_2, 'Quantity']
)

In [16]:
# Final Item Fix
item_map = {
    2: 'Coffee',
    1: 'Cookie',
    5: 'Salad',
    1.5: 'Tea'
}

item_mask = df['Item'] == 'Unknown'

df.loc[item_mask, 'Item'] = (
    df.loc[item_mask, 'Price Per Unit']
    .map(item_map)
    .fillna('Unknown')
)

In [17]:
# Remove NaN in Numerical Columns
cafe_sales = df[
    (df['Quantity'].notna())&
    (df['Price Per Unit'].notna())&
    (df['Total Spent'].notna())
]

In [18]:
# Feature Engineering
cafe_sales['day'] = cafe_sales['Transaction Date'].dt.strftime('%A')
cafe_sales['month'] = cafe_sales['Transaction Date'].dt.strftime('%B')

In [19]:
cafe_sales.shape

(9974, 10)

In [20]:
cafe_sales.info()

<class 'pandas.DataFrame'>
Index: 9974 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    9974 non-null   str           
 1   Item              9974 non-null   str           
 2   Quantity          9974 non-null   Int64         
 3   Price Per Unit    9974 non-null   float64       
 4   Total Spent       9974 non-null   float64       
 5   Payment Method    9974 non-null   str           
 6   Location          9974 non-null   str           
 7   Transaction Date  9514 non-null   datetime64[us]
 8   day               9514 non-null   str           
 9   month             9514 non-null   str           
dtypes: Int64(1), datetime64[us](1), float64(2), str(6)
memory usage: 866.9 KB


In [21]:
categorical_cols = [
    'Item',
    'Payment Method',
    'Location'
]

for col in categorical_cols:
    print(f'\n{col}')
    print(cafe_sales[col].unique())


Item
<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'Unknown',
 'Sandwich',      'Tea',    'Juice']
Length: 9, dtype: str

Payment Method
<StringArray>
['Credit Card', 'Cash', 'Unknown', 'Digital Wallet']
Length: 4, dtype: str

Location
<StringArray>
['Takeaway', 'In-store', 'Unknown']
Length: 3, dtype: str


In [22]:
cols = [
    'Item',
    'Quantity',
    'Price Per Unit',
    'Total Spent',
    'Payment Method',
    'Location',
    'Transaction Date'
]

summary = pd.DataFrame({
    'Unknown':(cafe_sales[cols]=='Unknown').sum(),
    'NaN':cafe_sales[cols].isna().sum()
}).T

summary

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
Unknown,474,0,0,0,3168,3952,0
NaN,0,0,0,0,0,0,460


In [23]:
cafe_sales = cafe_sales.rename(columns={
    'Transaction ID':'transaction_id',
    'Item':'item',
    'Quantity':'quantity',
    'Price Per Unit':'price_per_unit',
    'Total Spent':'total_spent',
    'Payment Method':'payment_method',
    'Location':'location',
    'Transaction Date':'transaction_date'
})

In [24]:
cafe_sales.head(10)

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,day,month
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,Friday,September
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,Tuesday,May
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19,Wednesday,July
3,TXN_7034554,Salad,2,5.0,10.0,Unknown,Unknown,2023-04-27,Thursday,April
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,Sunday,June
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,Unknown,2023-03-31,Friday,March
6,TXN_4433211,Unknown,3,3.0,9.0,Unknown,Takeaway,2023-10-06,Friday,October
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,Unknown,2023-10-28,Saturday,October
8,TXN_4717867,Unknown,5,3.0,15.0,Unknown,Takeaway,2023-07-28,Friday,July
9,TXN_2064365,Sandwich,5,4.0,20.0,Unknown,In-store,2023-12-31,Sunday,December
